# SIADS 516: Homework 3

- **Dr. Chris Teplovs**, School of Information, University of Michigan
- **Kris Steinhoff**, School of Information, University of Michigan


---
### Troubleshooting Tips

If you see `ConnectionRefused` errors from Spark, this usually means that the Spark server has crashed. A common reason for this is the Spark server running out of memory.

- Remember to avoid using `collect()` on large datasets!
- You can restart your notebook kernel (from the "Kernel" menu) to reset the Spark server.
---

This homework assignment builds on the Spark DataFrame material we covered in class.

You will be using a compressed version of the Yelp Academic Dataset.  The data set is provided for you in the assets/data/yelp_academic of your workspace and you should not need to download it again if you're working on the Coursera hosted notebook environment.

You might want to refer to the lecture companion notebooks (in resources/lecture_notebooks/ or equivalently via Coursera as "Ungraded Lab: Spark Core Demo" and "Ungraded Lab: Spark SQL Demo) for hints about libraries to import, etc.

You will notice that there are a **lot** of reviews.  You might want to work off a small sample (i.e. use the sample() function in Spark) to work on a reduced size dataset while you're developing your solution.

In [1]:
# The AutograderHelper class provides methods used by the autograder.
from autograder_helper import AutograderHelper
from mads.lib.path import assets

In [2]:
# Autograder cell. This cell is worth 0 points.
# This cell has hidden code used to configure the autograder.

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master("local[*]")
    .appName("My First Spark application")
    .getOrCreate()
)
sc = spark.sparkContext
sc._conf.set("spark.default.parallelism", 2)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/13 15:13:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# Set up some DataFrames:
file_user = assets.find("yelp_academic/yelp_academic_dataset_user.json.gz")
file_review = assets.find("yelp_academic/yelp_academic_dataset_review.json.gz")
file_checkin = assets.find("yelp_academic/yelp_academic_dataset_checkin.json.gz")

user = spark.read.json(str(file_user))
review = spark.read.json(str(file_review))
checkin = spark.read.json(str(file_checkin))

26/08/13 15:13:19 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

## Question 1 -- Cool Compliments

Determine how many users have received more than 5000 "cool" compliments.

- Create a variable `user_count` (an integer) which contains the number of user with more than 5000 "cool" compliments (using the `compliment_cool` field.)

In [5]:
# YOUR CODE HERE
#user.select("compliment_cool").show()
user_count = user.filter(user["compliment_cool"] > 5000).count()
#cool_users.select("compliment_cool").show()
#user_count = cool_users.count() 
print(user_count)

[Stage 3:>                                                          (0 + 1) / 1]

79


In [6]:
assert isinstance(user_count, int), "The user_count variable should be an integer."

In [7]:
# Autograder cell. This cell is worth 2 points (out of 20). This cell contains hidden tests.

## Question 2 -- Useful Positive Reviews

Determine the top 5 most useful positive reviews.

- Create a variable `top_5_useful_positive`. This should be a PySpark DataFrame
- For this question a "positive review" is one with 4 or 5 stars
- The DataFrame should be ordered by `useful` and contain 5 rows
- The DataFrame should have these columns (in this order):
    - `review_id`
    - `useful`
    - `stars`

In [8]:
# YOUR CODE HERE
top_5_useful_positive = review.filter(review["stars"] >= 4).sort("useful", ascending=False).limit(5).select("review_id", "useful", "stars")

#create pyspark dataframe
#filter review 4 or 5 stars
#sort by useful
#grab only top 5 rows
#limit column list

In [9]:
import pyspark

assert (
    type(top_5_useful_positive) == pyspark.sql.dataframe.DataFrame
), "The top_useful_positive variable should be a Spark DataFrame."

assert top_5_useful_positive.columns == [
    "review_id",
    "useful",
    "stars",
], "The columns are not in the correct order."

submitted = AutograderHelper.parse_spark_dataframe(top_5_useful_positive)

In [10]:
# Autograder cell. This cell is worth 1 point (out of 20). This cell does not contain hidden tests.
# This cell deliberately includes answers to provide guidance on how this question is graded.

assert len(submitted) == 5, "The result must have 5 rows."

top_useful_review_id = "1lGXlyq4MALOMx17vpBcoQ"
assert (
    submitted["review_id"][0] == top_useful_review_id
), f'The first row should have review_id "{top_useful_review_id}" (this review has the most "useful" ratings)'

In [11]:
# Autograder cell. This cell is worth 4 points (out of 20). This cell contains hidden tests.

## Question 3 -- Checkins

Determine what hours of the day most checkins occur.

- Create a variable `hours_by_checkin_count`. This should be a PySpark DataFrame
- The DataFrame should be ordered by `count` and contain 24 rows
- The DataFrame should have these columns (in this order):
    - `hour` (the hour of the day as an integer, the hour after midnight being `0`)
    - `count` (the number of checkins that occurred in that hour)
- Hour `1` (the time between 1:00 AM and 2:00 AM) should be the first entry in the results once they are sorted by the number of checkins that occurred in each hour.


Note that the `date` column in the `checkin` data is a string with multiple date times in it. You'll need to split that string before parsing.

In [12]:
checkin.first()

Row(business_id='--1UhMGODdWsrMastO9DZw', date='2016-04-26 19:49:16, 2016-08-30 18:36:57, 2016-10-15 02:45:18, 2016-11-18 01:54:50, 2017-04-20 18:39:06, 2017-05-03 17:58:02')

In [13]:
# YOUR CODE HERE
from pyspark.sql.functions import explode, split, to_timestamp, hour, trim

#df = checkin.withColumn("date", explode(split("date", ",")))
#df = df.withColumn("date", trim("date"))
#df = df.withColumn("date", to_timestamp("date", "yyyy-MM-dd HH:mm:ss"))
#df = df.withColumn("hour", hour("date"))

#hours_by_checkin_count = df.groupBy("hour").count().sort("count", ascending=False)

#hours_by_checkin_count.show()

In [14]:
#from pyspark.sql.functions import explode, split, to_timestamp, hour, trim

hours_by_checkin_count = (
      checkin.withColumn("date", explode(split("date", ",")))
      .withColumn("date", trim("date"))
      .withColumn("date", to_timestamp("date", "yyyy-MM-dd HH:mm:ss"))
      .withColumn("hour", hour("date"))
      .groupBy("hour")
      .count()
      .sort("count", ascending=False)
)

#hours_by_checkin_count = df.groupBy("hour").count().sort("count", ascending=False)

hours_by_checkin_count.show()

[Stage 8:>                                                          (0 + 1) / 1]

+----+-------+
|hour|  count|
+----+-------+
|   1|1561788|
|  19|1502271|
|   0|1491176|
|   2|1411255|
|  20|1350195|
|  23|1344117|
|  18|1272108|
|  22|1257437|
|  21|1238808|
|   3|1078939|
|  17|1006102|
|  16| 852076|
|   4| 747453|
|  15| 617830|
|   5| 485129|
|  14| 418340|
|   6| 321764|
|  13| 270145|
|   7| 231417|
|  12| 178910|
+----+-------+
only showing top 20 rows



In [15]:
assert (
    type(hours_by_checkin_count) == pyspark.sql.dataframe.DataFrame
), "The hours_by_checkin_count variable should be a Spark DataFrame."

assert hours_by_checkin_count.columns == [
    "hour",
    "count",
], "The columns are not in the correct order."

submitted = AutograderHelper.parse_spark_dataframe(hours_by_checkin_count)

In [16]:
# Autograder cell. This cell is worth 1 point (out of 20). This cell does not contain hidden tests.

assert len(submitted) == 24, "The hours_by_checkin_count DataFrame must have 24 rows."

assert submitted["hour"][0] == 1, "The first row should have hour 1"
assert submitted["hour"][1] == 19, "The second row should have hour 19"

In [17]:
# Autograder cell. This cell is worth 4 points (out of 20). This cell contains hidden tests.

## Question 4 -- Common Words in Useful Reviews

Write a function that takes a Spark DataFrame as a parameter and returns a Spark DataFrame of the 50 most common words from *useful* reviews and their counts.

- A "useful review" has 10 or more "useful" ratings.
- Convert the text to lower case.
- Use the provided `splitter()` function in a UDF to split the text into individual words.
- Exclude the words in the provided `STOP_WORDS` set.
- Returned DataFrame should have these columns (in this order):
    - `word`
    - `count`
- Returned DataFrame should be sorted by `count` in descending order.

In [18]:
review.first()

Row(business_id='ujmEBvifdJM6h6RLv4wQIg', cool=0, date='2013-05-07 04:34:36', funny=1, review_id='Q1sbwvVQXV2734tPgoKj4Q', stars=1.0, text='Total bill for this horrible service? Over $8Gs. These crooks actually had the nerve to charge us $69 for 3 pills. I checked online the pills can be had for 19 cents EACH! Avoid Hospital ERs at all costs.', useful=6, user_id='hG7b0MtEbXx5QzbzE6C_VA')

In [28]:
import re
from pyspark.sql.functions import udf, lower, explode, col
from pyspark.sql.types import ArrayType, StringType

def splitter(text):
    WORD_RE = re.compile(r"[\w']+")
    return WORD_RE.findall(text)


STOP_WORDS = {
    "a", "about", "above", "after", "again", "against", "aint", "all", "also",
    "although", "am", "an", "and", "any", "are", "as", "at", "be", "because",
    "been", "before", "being", "below", "between", "both", "but", "by", "can",
    "check", "checked", "could", "did", "do", "does", "doing", "don", "down",
    "during", "each", "few", "for", "from", "further", "get", "go", "got",
    "had", "has", "have", "having", "he", "her", "here", "hers", "herself",
    "him", "himself", "his", "how", "however", "i", "i'd", "if", "i'm", "in",
    "into", "is", "it", "its", "it's", "itself", "i've", "just", "me", "more",
    "most", "my", "myself", "no", "nor", "not", "now", "of", "off", "on",
    "once", "one", "online", "only", "or", "other", "our", "ours", "ourselves",
    "out", "over", "own", "paid", "place", "s", "said", "same", "service",
    "she", "should", "so", "some", "such", "t", "than", "that", "the", "their",
    "theirs", "them", "themselves", "then", "there", "these", "they", "this",
    "those", "through", "to", "too", "under", "until", "up", "us", "very",
    "was", "we", "went", "were", "we've", "what", "when", "where", "which",
    "while", "who", "whom", "why", "will", "with", "would", "you", "your",
    "yours", "yourself", "yourselves",
}


def common_useful_words(reviews, limit=50):
    # YOUR CODE HERE
    reviews = reviews.filter(reviews["useful"] >= 10)
    reviews = reviews.withColumn("text", lower("text"))
    textsplit = udf(lambda x: splitter(x), ArrayType(StringType()))
    most_common = (
        reviews.select(textsplit("text").alias("words"))
        .withColumn("word", explode("words"))
        .filter(~col("word").isin(STOP_WORDS))
        .groupBy("word")
        .count()
        .sort("count", ascending=False)
        .limit(limit)
    ) 
    return most_common

Now we'll run it on the `review` DataFrame

In [29]:
common_useful_words_counts = common_useful_words(review)

In [31]:
common_useful_words_counts.show()

[Stage 26:>                                                         (0 + 1) / 1]

+------+------+
|  word| count|
+------+------+
|  like|101251|
|  time| 86124|
|  good| 83486|
|  back| 71308|
|  food| 65281|
|  even| 58499|
|really| 57687|
| don't| 56146|
| great| 55402|
|  well| 48297|
|didn't| 45751|
| first| 43738|
|people| 42768|
|  know| 40954|
| never| 40741|
|     2| 39573|
|  told| 39350|
|   day| 38164|
|  came| 38098|
|  much| 37227|
+------+------+
only showing top 20 rows



In [30]:
assert (
    type(common_useful_words_counts) == pyspark.sql.dataframe.DataFrame
), "The common_useful_words_counts variable should be a Spark DataFrame."

assert common_useful_words_counts.columns == [
    "word",
    "count",
], "The columns are not in the correct order."

submitted = AutograderHelper.parse_spark_dataframe(common_useful_words_counts)

In [32]:
# Autograder cell. This cell is worth 2 points (out of 20). This cell does not contain hidden tests.

assert (
    len(submitted) == 50
), "The common_useful_words_counts DataFrame must have 50 rows."

assert submitted["word"][0] == "like", 'The first row should have word "like"'

assert submitted["count"][0] == 101251, "The first row should have count 101251"

In [33]:
# Autograder cell. This cell is worth 6 points (out of 20). This cell contains hidden tests.